# Recommendation Systems


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
file_path = r"D:\Desktop\Ausify Technologies Internship\Task 1 Data Cleaning\Output\netflix_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (8790, 12)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,90,min
1,s3,Tv Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",1,Season
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",1,Season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",125,min


In [3]:
print("Available Columns:")
print(df.columns.tolist())

print("\nRequired Columns:")
print(df[['title', 'type', 'listed_in']].head(10))

print("\nMissing Values:")
print(df[['title', 'type', 'listed_in']].isnull().sum())

Available Columns:
['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'duration_value', 'duration_unit']

Required Columns:
                              title     type  \
0              Dick Johnson Is Dead    Movie   
1                         Ganglands  Tv Show   
2                     Midnight Mass  Tv Show   
3  Confessions of an Invisible Girl    Movie   
4                           Sankofa    Movie   
5     The Great British Baking Show  Tv Show   
6                      The Starling    Movie   
7   Motu Patlu in the Game of Zones    Movie   
8                      Je Suis Karl    Movie   
9          Motu Patlu in Wonderland    Movie   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dramas, TV Horror, TV Mysteries  
3                 Children & Family Movies, Comedies  
4   Dr

In [4]:
# Select relevant columns
recommend_df = df[['title', 'type', 'listed_in']].copy()

# Remove records with missing essential information
recommend_df = recommend_df.dropna(
    subset=['title', 'type', 'listed_in']
)

# Create combined content features
recommend_df['content_features'] = (
    recommend_df['type'].astype(str) + ' ' +
    recommend_df['listed_in'].astype(str)
)

print("Content features created successfully!")

display(recommend_df.head(10))

Content features created successfully!


,title,type,listed_in,content_features
0,Dick Johnson Is Dead,Movie,Documentaries,Movie Documentaries
1,Ganglands,Tv Show,"Crime TV Shows, International TV Shows, TV Act...","Tv Show Crime TV Shows, International TV Shows..."
2,Midnight Mass,Tv Show,"TV Dramas, TV Horror, TV Mysteries","Tv Show TV Dramas, TV Horror, TV Mysteries"
3,Confessions of an Invisible Girl,Movie,"Children & Family Movies, Comedies","Movie Children & Family Movies, Comedies"
4,Sankofa,Movie,"Dramas, Independent Movies, International Movies","Movie Dramas, Independent Movies, Internationa..."
5,The Great British Baking Show,Tv Show,"British TV Shows, Reality TV","Tv Show British TV Shows, Reality TV"
6,The Starling,Movie,"Comedies, Dramas","Movie Comedies, Dramas"
7,Motu Patlu in the Game of Zones,Movie,"Children & Family Movies, Comedies, Music & Mu...","Movie Children & Family Movies, Comedies, Musi..."
8,Je Suis Karl,Movie,"Dramas, International Movies","Movie Dramas, International Movies"
9,Motu Patlu in Wonderland,Movie,"Children & Family Movies, Music & Musicals","Movie Children & Family Movies, Music & Musicals"


In [5]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


recommend_df['cleaned_features'] = (
    recommend_df['content_features'].apply(preprocess_text)
)

display(
    recommend_df[['title', 'content_features', 'cleaned_features']].head(10)
)

,title,content_features,cleaned_features
0,Dick Johnson Is Dead,Movie Documentaries,movie documentaries
1,Ganglands,"Tv Show Crime TV Shows, International TV Shows...",tv show crime tv shows international tv shows ...
2,Midnight Mass,"Tv Show TV Dramas, TV Horror, TV Mysteries",tv show tv dramas tv horror tv mysteries
3,Confessions of an Invisible Girl,"Movie Children & Family Movies, Comedies",movie children family movies comedies
4,Sankofa,"Movie Dramas, Independent Movies, Internationa...",movie dramas independent movies international ...
5,The Great British Baking Show,"Tv Show British TV Shows, Reality TV",tv show british tv shows reality tv
6,The Starling,"Movie Comedies, Dramas",movie comedies dramas
7,Motu Patlu in the Game of Zones,"Movie Children & Family Movies, Comedies, Musi...",movie children family movies comedies music mu...
8,Je Suis Karl,"Movie Dramas, International Movies",movie dramas international movies
9,Motu Patlu in Wonderland,"Movie Children & Family Movies, Music & Musicals",movie children family movies music musicals


In [6]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer()

# Convert cleaned text into numerical vectors
tfidf_matrix = tfidf.fit_transform(
    recommend_df['cleaned_features']
)

print("TF-IDF Matrix Created Successfully!")
print("Matrix Shape:", tfidf_matrix.shape)
print("Number of Features:", len(tfidf.get_feature_names_out()))

TF-IDF Matrix Created Successfully!
Matrix Shape: (8790, 47)
Number of Features: 47


In [7]:
# Calculate cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Calculated Successfully!")
print("Similarity Matrix Shape:", cosine_sim.shape)

Cosine Similarity Calculated Successfully!
Similarity Matrix Shape: (8790, 8790)


In [8]:
# Create title index mapping
indices = pd.Series(
    recommend_df.index,
    index=recommend_df['title']
).drop_duplicates()


def recommend_titles(title, top_n=5):

    if title not in indices:
        return "Title not found in the Netflix dataset."

    idx = indices[title]

    # Get similarity scores
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude the selected title itself
    similarity_scores = [
        score for score in similarity_scores if score[0] != idx
    ]

    # Select top recommendations
    top_matches = similarity_scores[:top_n]

    # Get recommended titles
    movie_indices = [i[0] for i in top_matches]

    results = recommend_df.iloc[movie_indices][
        ['title', 'type', 'listed_in']
    ].copy()

    results['similarity_score'] = [
        round(score[1], 3) for score in top_matches
    ]

    return results.reset_index(drop=True)

In [9]:
recommend_titles('The Starling', top_n=5)

,title,type,listed_in,similarity_score
0,Ginny & Georgia - The Afterparty,Movie,"Comedies, Dramas",1.0
1,The Edge of Seventeen,Movie,"Comedies, Dramas",1.0
2,Mother's Day,Movie,"Comedies, Dramas",1.0
3,War Dogs,Movie,"Comedies, Dramas",1.0
4,The Disaster Artist,Movie,"Comedies, Dramas",1.0


In [10]:
test_titles = ['The Starling', 'Sankofa', 'Midnight Mass']

for title in test_titles:
    print("\nRecommendations for:", title)
    recommendations = recommend_titles(title, top_n=5)
    display(recommendations)


Recommendations for: The Starling


,title,type,listed_in,similarity_score
0,Ginny & Georgia - The Afterparty,Movie,"Comedies, Dramas",1.0
1,The Edge of Seventeen,Movie,"Comedies, Dramas",1.0
2,Mother's Day,Movie,"Comedies, Dramas",1.0
3,War Dogs,Movie,"Comedies, Dramas",1.0
4,The Disaster Artist,Movie,"Comedies, Dramas",1.0



Recommendations for: Sankofa


,title,type,listed_in,similarity_score
0,Ankahi Kahaniya,Movie,"Dramas, Independent Movies, International Movies",1.0
1,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",1.0
2,La diosa del asfalto,Movie,"Dramas, Independent Movies, International Movies",1.0
3,Eyes of a Thief,Movie,"Dramas, Independent Movies, International Movies",1.0
4,Pomegranates and Myrrh,Movie,"Dramas, Independent Movies, International Movies",1.0



Recommendations for: Midnight Mass


,title,type,listed_in,similarity_score
0,Brand New Cherry Flavor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",1.0
1,The Haunting of Bly Manor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",1.0
2,Ratched,Tv Show,"TV Dramas, TV Horror, TV Mysteries",1.0
3,The Haunting of Hill House,Tv Show,"TV Dramas, TV Horror, TV Mysteries",1.0
4,The Originals,Tv Show,"TV Dramas, TV Horror, TV Mysteries",1.0


In [11]:
# Test 1: Recommendations should not include the original title
# Test 2: Similarity scores should be in descending order
# Test 3: Recommendations should contain the required columns

recommendations = recommend_titles('The Starling', top_n=5)

print("Original title excluded:",
      'The Starling' not in recommendations['title'].values)

print("Scores in descending order:",
      recommendations['similarity_score'].is_monotonic_decreasing)

print("Required columns present:",
      all(col in recommendations.columns
          for col in ['title', 'type', 'listed_in', 'similarity_score']))

print("\nUnknown title test:")
print(recommend_titles('Random Nonexistent Title'))

Original title excluded: True
Scores in descending order: True
Required columns present: True

Unknown title test:
Title not found in the Netflix dataset.


### Evaluation of Recommendation System

The content-based recommendation system was evaluated using three sample Netflix titles: *The Starling*, *Sankofa*, and *Midnight Mass*.

The following checks were performed:

1. **Recommendation relevance:** Recommended titles were compared based on their content categories and type.
2. **Self-recommendation check:** The system successfully excluded the input title from its recommendations.
3. **Similarity ranking:** Recommendations were returned in descending order of cosine similarity scores.
4. **Invalid title handling:** The system correctly displayed a message when a title was not found in the dataset.

All implemented validation checks passed successfully.

**Limitations:** The model uses only title type and genre categories because the dataset does not contain plot descriptions or user ratings. Therefore, some recommendations may have identical similarity scores. Formal precision and recall evaluation was not possible because labelled user preferences were unavailable.

### Conclusion

A basic content-based recommendation system was successfully developed using the Netflix dataset.

TF-IDF Vectorization was used to convert content categories into numerical features, and Cosine Similarity was applied to identify titles with similar characteristics.

The system can generate recommendations for selected Netflix titles, rank them by similarity, exclude the original title, and handle invalid title inputs.

This project provided practical experience with text preprocessing, feature extraction, TF-IDF, cosine similarity, and recommendation systems.

Future improvements could include plot descriptions, cast information, director details, user ratings, and user preference data to generate more diverse and personalized recommendations.